In [1]:
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.1+cu130
GPU available: True
GPU name: Tesla T4


In [3]:
%pip install -q pymupdf sentence-transformers faiss-cpu streamlit python-dotenv neo4j


In [4]:
import pymupdf
import sentence_transformers
import faiss
import streamlit
import dotenv
import neo4j
import torch

print("All libraries imported successfully!")
print("GPU available:", torch.cuda.is_available())

All libraries imported successfully!
GPU available: True


In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pathlib import Path

# Common project directory
PROJECT = Path("/content/drive/MyDrive/KG-RAG-Agriculture")

# Verify project directory
if not PROJECT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT}"
    )

print("Project directory:", PROJECT)
print("Project setup successful!")

Project directory: /content/drive/MyDrive/KG-RAG-Agriculture
Project setup successful!


In [7]:
!ls -la /content/drive/MyDrive/KG-RAG-Agriculture


lrw------- 1 root root 0 Nov 18 19:07 /content/drive/MyDrive/KG-RAG-Agriculture -> /content/drive/.shortcut-targets-by-id/1n27ULkSTd8Au8etDQa8ydkYFjBnLQapT/KG-RAG-Agriculture


In [8]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/KG-RAG-Agriculture")

# Create additional folders
new_folders = [
    "outputs",
    "answers",
    "evaluation_results",
    "logs",
    "frontend"
]

for folder in new_folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

# Create project files if they don't already exist
new_files = [
    "app.py",
    "requirements.txt",
    "README.md",
    ".gitignore"
]

for file in new_files:
    file_path = PROJECT / file

    if not file_path.exists():
        file_path.touch()

print("All additional folders and files created successfully!")

All additional folders and files created successfully!


In [9]:
for path in sorted(PROJECT.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(PROJECT))

📁 answers
📁 data
📁 data/books
📁 data/metadata
📁 data/processed
📁 data/raw
📁 data/reports
📁 data/research_papers
📁 evaluation
📁 evaluation_results
📁 frontend
📁 knowledge_graph
📁 knowledge_graph/exports
📁 knowledge_graph/schemas
📁 logs
📁 notebooks
📁 outputs
📁 src
📁 src/chunking
📁 src/embeddings
📁 src/evaluation
📁 src/generation
📁 src/graph
📁 src/ingestion
📁 src/retrieval
📁 src/vector_store
📁 vector_store
📁 vector_store/faiss_index


In [10]:
for item in sorted(PROJECT.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

📄 .gitignore
📄 README.md
📁 answers
📄 app.py
📁 data
📁 evaluation
📁 evaluation_results
📁 frontend
📁 knowledge_graph
📁 logs
📁 notebooks
📁 outputs
📄 requirements.txt
📁 src
📁 vector_store


In [11]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/KG-RAG-Agriculture")

print("Project folder found:", PROJECT.exists())

Project folder found: True


In [12]:
#PDF TEXT EXTRACTION
import fitz

pdf_path = "/content/drive/MyDrive/KG-RAG-Agriculture/data/research_papers/IRRI_Foliar_Fungal_Diseases.pdf"

doc = fitz.open(pdf_path)

print("Total pages:", len(doc))

for page_number in range(min(3, len(doc))):
    print(f"\n--- Page {page_number + 1} ---")
    print(doc[page_number].get_text()[:2000])

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Total pages: 115

--- Page 1 ---
1
Chapter 2. Foliar fungal diseases
Throughout history, among the rice diseases, foliar diseases have imposed the most seri­
ous threat to rice production. Both rice blast and brown spot, the two most recorded rice 
diseases, have caused significant damage to rice production that has resulted in human 
catastrophes. Blast is well known and feared by rice farmers because of its explosive dam­
aging potential. Blast epidemics have often destroyed rice harvests, sometimes resulting 
in socioeconomic as well as political uncertainty in some nations. Brown spot is the direct 
cause of the “Great Bengal Famine (see below)” and has been known to be consistently 
associated with poor rice crop production environments, especially in rice fields deficient 
in nutrients, particularly nitrogen (Ou 1985).  
	
Because of blast’s threat, over the past three decades, researchers have used this 
disease as a “model system” in plant molecular biology and biotechnology. R

In [13]:
#Combines all extracted text and saves into one text file
import pymupdf
from pathlib import Path

pdf_path = Path(
    "/content/drive/MyDrive/KG-RAG-Agriculture/data/research_papers/IRRI_Foliar_Fungal_Diseases.pdf"
)

output_folder = Path(
    "/content/drive/MyDrive/KG-RAG-Agriculture/data/processed"
)

output_folder.mkdir(parents=True, exist_ok=True)

doc = pymupdf.open(pdf_path)

full_text = ""

for page_number, page in enumerate(doc):
    full_text += f"\n\n--- Page {page_number + 1} ---\n"
    full_text += page.get_text()

output_path = output_folder / "IRRI_Foliar_Fungal_Diseases.txt"

output_path.write_text(full_text, encoding="utf-8")

print("Total pages processed:", len(doc))
print("Total characters extracted:", len(full_text))
print("Text saved successfully at:", output_path)

Total pages processed: 115
Total characters extracted: 333191
Text saved successfully at: /content/drive/MyDrive/KG-RAG-Agriculture/data/processed/IRRI_Foliar_Fungal_Diseases.txt


In [14]:
#Check the extracted text
text_file = output_folder / "IRRI_Foliar_Fungal_Diseases.txt"

sample_text = text_file.read_text(encoding="utf-8")

print(sample_text[:3000])



--- Page 1 ---
1
Chapter 2. Foliar fungal diseases
Throughout history, among the rice diseases, foliar diseases have imposed the most seri­
ous threat to rice production. Both rice blast and brown spot, the two most recorded rice 
diseases, have caused significant damage to rice production that has resulted in human 
catastrophes. Blast is well known and feared by rice farmers because of its explosive dam­
aging potential. Blast epidemics have often destroyed rice harvests, sometimes resulting 
in socioeconomic as well as political uncertainty in some nations. Brown spot is the direct 
cause of the “Great Bengal Famine (see below)” and has been known to be consistently 
associated with poor rice crop production environments, especially in rice fields deficient 
in nutrients, particularly nitrogen (Ou 1985).  
	
Because of blast’s threat, over the past three decades, researchers have used this 
disease as a “model system” in plant molecular biology and biotechnology. Research has 
foc

In [15]:
#Cleaning: remove repeated spaces and tabs, reduce excessive blank lines.
import re

# Read the extracted text
text_file = output_folder / "IRRI_Foliar_Fungal_Diseases.txt"
raw_text = text_file.read_text(encoding="utf-8")

# Clean unnecessary spaces and blank lines
clean_text = re.sub(r'[ \t]+', ' ', raw_text)
clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)
clean_text = clean_text.strip()

# Save the cleaned text
clean_file = output_folder / "IRRI_Foliar_Fungal_Diseases_cleaned.txt"
clean_file.write_text(clean_text, encoding="utf-8")

print("Original characters:", len(raw_text))
print("Cleaned characters:", len(clean_text))
print("Cleaned text saved at:", clean_file)

Original characters: 333191
Cleaned characters: 332335
Cleaned text saved at: /content/drive/MyDrive/KG-RAG-Agriculture/data/processed/IRRI_Foliar_Fungal_Diseases_cleaned.txt


In [16]:
#Text chunking
# Divide the cleaned text into chunks

chunk_size = 1000
overlap = 200

chunks = []

start = 0

while start < len(clean_text):
    end = start + chunk_size

    chunk = clean_text[start:end].strip()

    if chunk:
        chunks.append(chunk)

    start += chunk_size - overlap

print("Total chunks created:", len(chunks))

print("\nFirst chunk:\n")
print(chunks[0])

Total chunks created: 416

First chunk:

--- Page 1 ---
1
Chapter 2. Foliar fungal diseases
Throughout history, among the rice diseases, foliar diseases have imposed the most seri­
ous threat to rice production. Both rice blast and brown spot, the two most recorded rice 
diseases, have caused significant damage to rice production that has resulted in human 
catastrophes. Blast is well known and feared by rice farmers because of its explosive dam­
aging potential. Blast epidemics have often destroyed rice harvests, sometimes resulting 
in socioeconomic as well as political uncertainty in some nations. Brown spot is the direct 
cause of the “Great Bengal Famine (see below)” and has been known to be consistently 
associated with poor rice crop production environments, especially in rice fields deficient 
in nutrients, particularly nitrogen (Ou 1985). 
 
Because of blast’s threat, over the past three decades, researchers have used this 
disease as a “model system” in plant molecular biolog

In [17]:
#Save chunks as JSON file
import json

# Prepare each chunk with a unique ID
chunk_data = []

for i, chunk in enumerate(chunks):
    chunk_data.append({
        "chunk_id": i,
        "text": chunk,
        "source": "IRRI_Foliar_Fungal_Diseases.pdf"
    })

# Save chunks in Google Drive
chunks_file = output_folder / "IRRI_Foliar_Fungal_Diseases_chunks.json"

with open(chunks_file, "w", encoding="utf-8") as f:
    json.dump(chunk_data, f, ensure_ascii=False, indent=4)

print("Total chunks saved:", len(chunk_data))
print("Saved file:", chunks_file)
print("\nFirst chunk:")
print(chunk_data[0]["text"])

Total chunks saved: 416
Saved file: /content/drive/MyDrive/KG-RAG-Agriculture/data/processed/IRRI_Foliar_Fungal_Diseases_chunks.json

First chunk:
--- Page 1 ---
1
Chapter 2. Foliar fungal diseases
Throughout history, among the rice diseases, foliar diseases have imposed the most seri­
ous threat to rice production. Both rice blast and brown spot, the two most recorded rice 
diseases, have caused significant damage to rice production that has resulted in human 
catastrophes. Blast is well known and feared by rice farmers because of its explosive dam­
aging potential. Blast epidemics have often destroyed rice harvests, sometimes resulting 
in socioeconomic as well as political uncertainty in some nations. Brown spot is the direct 
cause of the “Great Bengal Famine (see below)” and has been known to be consistently 
associated with poor rice crop production environments, especially in rice fields deficient 
in nutrients, particularly nitrogen (Ou 1985). 
 
Because of blast’s threat, over

In [18]:
#Generate embeddings
from sentence_transformers import SentenceTransformer
import numpy as np
import json

# Load the saved chunks
with open(chunks_file, "r", encoding="utf-8") as f:
    chunk_data = json.load(f)

# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract text from chunks
texts = [item["text"] for item in chunk_data]

# Generate embeddings
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Total chunks:", len(texts))
print("Embedding shape:", embeddings.shape)
print("First embedding (first 10 values):")
print(embeddings[0][:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Total chunks: 416
Embedding shape: (416, 384)
First embedding (first 10 values):
[-0.00174297  0.02208242 -0.05240539  0.02099428  0.0587182   0.00088711
 -0.02903642  0.06049155  0.02673022 -0.01072977]


In [19]:
#Save the embeddings
import numpy as np

# Save embeddings
embedding_file = output_folder / "IRRI_Foliar_Fungal_Diseases_embeddings.npy"

np.save(embedding_file, embeddings)

print("Embeddings saved successfully!")
print("File location:", embedding_file)
print("Embedding shape:", embeddings.shape)

Embeddings saved successfully!
File location: /content/drive/MyDrive/KG-RAG-Agriculture/data/processed/IRRI_Foliar_Fungal_Diseases_embeddings.npy
Embedding shape: (416, 384)


In [20]:
#Create FAISS index
import faiss
import numpy as np

# Convert embeddings to FAISS-compatible format
embeddings = np.asarray(embeddings, dtype="float32")

# Create a FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)
print("Embedding dimensions:", index.d)

FAISS index created successfully!
Total vectors stored: 416
Embedding dimensions: 384


In [21]:
#Save FAISS index
faiss_file = output_folder / "IRRI_Foliar_Fungal_Diseases.faiss"

faiss.write_index(index, str(faiss_file))

print("FAISS index saved successfully!")
print("Saved at:", faiss_file)

FAISS index saved successfully!
Saved at: /content/drive/MyDrive/KG-RAG-Agriculture/data/processed/IRRI_Foliar_Fungal_Diseases.faiss


In [22]:
#Testing semantic search
# Test semantic search using FAISS

query = "What are the symptoms of rice blast disease?"

# Convert the question into an embedding
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

# Search for the 5 most relevant chunks
distances, indices = index.search(query_embedding, 5)

# Display the retrieved results
print("User Question:", query)
print("\nTop 5 Relevant Results:\n")

for rank, idx in enumerate(indices[0]):
    print(f"\nResult {rank + 1}")
    print("Distance:", distances[0][rank])
    print("Chunk ID:", chunk_data[idx]["chunk_id"])
    print("Text:")
    print(chunk_data[idx]["text"])
    print("-" * 70)

User Question: What are the symptoms of rice blast disease?

Top 5 Relevant Results:


Result 1
Distance: 0.6364659
Chunk ID: 65
Text:
to its occurrence at low temperatures under field conditions. The active temperature 
range for disease development is 15-25oC with an optimum of about 20oC or slightly lower 
(Yamanaka and Kawai 1961). The disease, therefore, occurs only in a temperate rice pro­
duction environment where the rice crop is likely to suffer stress due to low temperature.
2.1. Symptoms
In the earlier stage of the disease, new leaves of infected plants produce chlorotic whitish 
spots or patches on the leaf blades, sometimes reducing them in size. At times, fluffy white 
to grey fungal growth may be seen on the under surface of the leaf blades. Infected young 
shoots and seeds may also show white coating of fungal growth.
 
In severe cases, the leaves may be distorted or twisted. The symptoms are more 
pronounced at flowering time. The panicles appear irregular, partially e

In [23]:
#Testing another question
query = "What are the symptoms and causes of brown spot disease in rice?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

distances, indices = index.search(query_embedding, 5)

print("Question:", query)

for rank, idx in enumerate(indices[0]):
    print(f"\nResult {rank + 1}")
    print("Distance:", distances[0][rank])
    print("Chunk ID:", chunk_data[idx]["chunk_id"])
    print(chunk_data[idx]["text"][:1000])
    print("-" * 60)

Question: What are the symptoms and causes of brown spot disease in rice?

Result 1
Distance: 0.5429125
Chunk ID: 10
ations of consumers are less than desirable. Finding ways to 
minimize losses due to rice diseases can immediately improve the household food securi­
ty of resource-poor rice farmers. 
1.2. Symptoms
The disease is found on all parts and growth stages of the rice plant from seeds to the 
whole plant, and from seedlings to mature plants (BSp Figure 2). BSp on rice leaves 
typically shows oval necrotic lesions surrounded by chlorotic halos. The typical oval spots 
on leaf blades are about the size 
and shape of the sesame seeds, 
hence, the name sesame leaf blight 
disease of rice given by Japanese 
scientists. The leaf spots are brown 
with gray or whitish centers when 
fully developed. On some rice 
varieties, the lesions may be large 
with a dark brown center. Young 
lesions are small and circular and 
may appear initially as dark brown 
or purplish brown dots on leaf 
b